In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
test = pd.read_csv("/Users/fesavaskurt/Python&ML/Kodlarım/Python/datasets/ML Datasets/ML Projects/GiveMeSomeCredit/cs-test.csv")
train = pd.read_csv("/Users/fesavaskurt/Python&ML/Kodlarım/Python/datasets/ML Datasets/ML Projects/GiveMeSomeCredit/cs-training.csv")

In [4]:
test.shape

(101503, 12)

In [5]:
train.shape

(150000, 12)

In [6]:
df = train.copy()

In [7]:
df.head()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines          150000 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 11  

In [10]:
df.describe()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,1.202690e+05,150000.000000,150000.000000,150000.000000,150000.000000,146076.000000
mean,75000.500000,0.066840,6.048438,52.295207,0.421033,353.005076,6.670221e+03,8.452760,0.265973,1.018240,0.240387,0.757222
std,43301.414527,0.249746,249.755371,14.771866,4.192781,2037.818523,1.438467e+04,5.145951,4.169304,1.129771,4.155179,1.115086
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,37500.750000,0.000000,0.029867,41.000000,0.000000,0.175074,3.400000e+03,5.000000,0.000000,0.000000,0.000000,0.000000
50%,75000.500000,0.000000,0.154181,52.000000,0.000000,0.366508,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
75%,112500.250000,0.000000,0.559046,63.000000,0.000000,0.868254,8.249000e+03,11.000000,0.000000,2.000000,0.000000,1.000000
max,150000.000000,1.000000,50708.000000,109.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000


In [12]:
df["SeriousDlqin2yrs"].value_counts(normalize= True)

SeriousDlqin2yrs
0    0.93316
1    0.06684
Name: proportion, dtype: float64

#### Dataset Yorumu
- Veri seti 150k gözlem ve 12 değişkenden oluşuyor. Hedef değişken SeriousDlqin2yrs olup, müşterilerin 2 yıl içerisinde defaulta düşme oranını gösterir. 
- Hedef değişken dağılımında %6.7 oranında pozitif sınıf gözükmektedir. Bu da veri setinin imbalanced olduğunu gösteirr. Bu nedenle model performansını değerlendirirken accuracy değil, ROC-AUC, Recall ve Precision gibi metrikler kullanılacaktır.
- Ayrıca MonthlyIncome ve NumberOfDependents değişkenlerinde eksik değerler gözlemlenmektedir. Bu eksiklerin rastgele olup olmadığı analiz edilecektir.

#### Kolonlar ve İncelemeleri
| Kolon                                    | Teknik Anlamı                                             | Bankacılık Yorumu                                   |
| ---------------------------------------- | --------------------------------------------------------- | --------------------------------------------------- |
| **SeriousDlqin2yrs**                     | Önümüzdeki 2 yıl içinde 90+ gün gecikmeye düştü mü? (0/1) | Hedef değişkenimiz. 1 = problemli müşteri           |
| **RevolvingUtilizationOfUnsecuredLines** | Kullanılan revolving limit / toplam revolving limit       | Kredi kartı limit kullanım oranı gibi düşünülebilir |
| **age**                                  | Müşteri yaşı                                              | Yaş arttıkça risk profili değişebilir               |
| **NumberOfTime30-59DaysPastDueNotWorse** | Son dönemde 30-59 gün arası gecikme sayısı                | En güçlü risk sinyallerinden biridir                |
| **DebtRatio**                            | Toplam borç ödeme yükü / gelir                            | Gelire göre borçluluk seviyesi                      |
| **MonthlyIncome**                        | Aylık gelir                                               | Müşterinin ödeme kapasitesi                         |
| **NumberOfOpenCreditLinesAndLoans**      | Açık kredi ve kredi hesabı sayısı                         | Bankacılık ilişkisi yoğunluğu                       |
| **NumberOfTimes90DaysLate**              | 90+ gün gecikme sayısı                                    | Çok kritik risk göstergesi                          |
| **NumberRealEstateLoansOrLines**         | Konut kredisi veya ipotekli kredi sayısı                  | Gayrimenkul kredi ilişkileri                        |
| **NumberOfTime60-89DaysPastDueNotWorse** | 60-89 gün arası gecikme sayısı                            | Temerrüde yaklaşan davranış                         |
| **NumberOfDependents**                   | Bakmakla yükümlü olunan kişi sayısı                       | Finansal yük göstergesi                             |
| **Unnamed: 0**                           | ID kolonu                                                 | Modele sokulmayacak                                 |


In [22]:
df.groupby(by= "SeriousDlqin2yrs")[["NumberOfTimes90DaysLate", 
                                    "NumberOfTime60-89DaysPastDueNotWorse",
                                   "NumberOfTime30-59DaysPastDueNotWorse",
                                   "RevolvingUtilizationOfUnsecuredLines",
                                   "DebtRatio"]].agg(["mean"])

,NumberOfTimes90DaysLate,NumberOfTime60-89DaysPastDueNotWorse,NumberOfTime30-59DaysPastDueNotWorse,RevolvingUtilizationOfUnsecuredLines,DebtRatio
,mean,mean,mean,mean,mean
SeriousDlqin2yrs,,,,,
0,0.135225,0.126666,0.280109,6.168855,357.151168
1,2.091362,1.828047,2.388490,4.367282,295.121066


In [24]:
for col in train.columns:
    if col not in ["SeriousDlqin2yrs", "Unnamed: 0"]:
        print(col)
        print(
            train.groupby("SeriousDlqin2yrs")[col].agg(["median", "mean"])
        )
        print("-"*50)

RevolvingUtilizationOfUnsecuredLines
                    median      mean
SeriousDlqin2yrs                    
0                 0.133288  6.168855
1                 0.838853  4.367282
--------------------------------------------------
age
                  median       mean
SeriousDlqin2yrs                   
0                   52.0  52.751375
1                   45.0  45.926591
--------------------------------------------------
NumberOfTime30-59DaysPastDueNotWorse
                  median      mean
SeriousDlqin2yrs                  
0                    0.0  0.280109
1                    0.0  2.388490
--------------------------------------------------
DebtRatio
                    median        mean
SeriousDlqin2yrs                      
0                 0.362659  357.151168
1                 0.428227  295.121066
--------------------------------------------------
MonthlyIncome
                  median         mean
SeriousDlqin2yrs                     
0                 5466.0  6747

In [45]:
df = df.drop("Unnamed: 0", axis = 1)

In [48]:
numeric_cols = df.select_dtypes(include=["number"]).columns

In [51]:
for col in numeric_cols:
    print(
        df.groupby("SeriousDlqin2yrs")[col].quantile([0.25, 0.5, 0.75])
    )
    print("-"*50)

SeriousDlqin2yrs      
0                 0.25    0.0
                  0.50    0.0
                  0.75    0.0
1                 0.25    1.0
                  0.50    1.0
                  0.75    1.0
Name: SeriousDlqin2yrs, dtype: float64
--------------------------------------------------
SeriousDlqin2yrs      
0                 0.25    0.026983
                  0.50    0.133288
                  0.75    0.487686
1                 0.25    0.398219
                  0.50    0.838853
                  0.75    1.000000
Name: RevolvingUtilizationOfUnsecuredLines, dtype: float64
--------------------------------------------------
SeriousDlqin2yrs      
0                 0.25    42.0
                  0.50    52.0
                  0.75    63.0
1                 0.25    36.0
                  0.50    45.0
                  0.75    54.0
Name: age, dtype: float64
--------------------------------------------------
SeriousDlqin2yrs      
0                 0.25    0.0
                  0.50   

In [53]:
for col in [
    "age",
    "MonthlyIncome",
    "RevolvingUtilizationOfUnsecuredLines",
    "DebtRatio"
]:
    df["BIN"] = pd.qcut(df[col], 10, duplicates= "drop")
    
    print(
        df.groupby("BIN")["SeriousDlqin2yrs"].agg(["count", "mean"])
    )
    print("-"*50)

                count      mean
BIN                            
(-0.001, 33.0]  17085  0.113550
(33.0, 39.0]    14919  0.095851
(39.0, 44.0]    15799  0.086398
(44.0, 48.0]    14741  0.081406
(48.0, 52.0]    14826  0.077027
(52.0, 56.0]    14214  0.065780
(56.0, 61.0]    16878  0.049710
(61.0, 65.0]    12939  0.037484
(65.0, 72.0]    14258  0.026652
(72.0, 109.0]   14341  0.021616
--------------------------------------------------
                      count      mean
BIN                                  
(-0.001, 2005.0]      12028  0.084386
(2005.0, 3000.0]      13056  0.096661
(3000.0, 3800.0]      11322  0.091150
(3800.0, 4544.2]      11702  0.080926
(4544.2, 5400.0]      12207  0.073319
(5400.0, 6300.0]      11850  0.066835
(6300.0, 7500.0]      12351  0.059266
(7500.0, 9083.0]      11733  0.051138
(9083.0, 11666.0]     12110  0.045169
(11666.0, 3008750.0]  11910  0.044920
--------------------------------------------------
                   count      mean
BIN                    

/var/folders/sj/pj9lwv9d1r17b6cggf8ym3gc0000gn/T/ipykernel_31585/2404556154.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("BIN")["SeriousDlqin2yrs"].agg(["count", "mean"])
/var/folders/sj/pj9lwv9d1r17b6cggf8ym3gc0000gn/T/ipykernel_31585/2404556154.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("BIN")["SeriousDlqin2yrs"].agg(["count", "mean"])
/var/folders/sj/pj9lwv9d1r17b6cggf8ym3gc0000gn/T/ipykernel_31585/2404556154.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain curre

RevolvingUtilizationOfUnsecuredLines değişkeni ile temerrüt oranı arasında güçlü ve pozitif ilişki gözlenmiştir. Limit kullanım oranı arttıkça temerrüt oranı belirgin şekilde yükselmektedir.

In [56]:
df["DebtRatio"].describe(percentiles=[0.90,0.95,0.99,0.995,0.9999])

count     150000.000000
mean         353.005076
std         2037.818523
min            0.000000
50%            0.366508
90%         1267.000000
95%         2449.000000
99%         4979.040000
99.5%       6186.010000
99.99%     40362.002300
max       329664.000000
Name: DebtRatio, dtype: float64

In [58]:
for col in ["MonthlyIncome","NumberOfDependents"]:

    print("\n", col)

    print(
        df.groupby(
            df[col].isnull()
        )["SeriousDlqin2yrs"]
        .agg(["count","mean"])
    )


 MonthlyIncome
                count      mean
MonthlyIncome                  
False          120269  0.069486
True            29731  0.056137

 NumberOfDependents
                     count      mean
NumberOfDependents                  
False               146076  0.067410
True                  3924  0.045617


### Model Kurulumu

In [59]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

In [64]:
X = df.drop(columns=["SeriousDlqin2yrs", "BIN"], axis = 1)
y = df["SeriousDlqin2yrs"]

In [65]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size= 0.2, stratify= y, random_state= 42
)

In [66]:
imputer = SimpleImputer(strategy= "mean")

In [67]:
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

In [68]:
scaler =StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

In [69]:
reg = LogisticRegression(
    max_iter = 10000,
    class_weight= "balanced",
    random_state= 42
)

In [70]:
reg.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,10000
,multi_class,'deprecated'


In [71]:
test_proba = reg.predict_proba(X_test_scaled)[:, 1]
test_pred = reg.predict(X_test_scaled)

In [73]:
print("ROC-AUC: ", roc_auc_score(y_test, test_pred))
print("Matrix: ", confusion_matrix(y_test, test_pred))
print("report: ", classification_report(y_test, test_pred))

ROC-AUC:  0.7265547508260961
Matrix:  [[21942  6053]
 [  663  1342]]
report:                precision    recall  f1-score   support

           0       0.97      0.78      0.87     27995
           1       0.18      0.67      0.29      2005

    accuracy                           0.78     30000
   macro avg       0.58      0.73      0.58     30000
weighted avg       0.92      0.78      0.83     30000



Ham değişkenlerle Logistic Regression belirli bir ayrıştırma gücü üretiyor. Ancak model çok fazla false positive üretiyor. Bu nedenle threshold tuning, median imputation, missing flag, outlier treatment ve gelişmiş modellerle iyileştirme yapılmalı.

In [74]:
X = df.drop(columns=["SeriousDlqin2yrs", "BIN"], axis = 1)
y = df["SeriousDlqin2yrs"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size= 0.2, stratify= y, random_state= 42
)

imputer = SimpleImputer(strategy= "mean")

X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

scaler =StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

reg = LogisticRegression(
    max_iter = 10000,
    class_weight= None,
    random_state= 42
)

reg.fit(X_train_scaled, y_train)

test_proba = reg.predict_proba(X_test_scaled)[:, 1]
test_pred = reg.predict(X_test_scaled)

print("ROC-AUC: ", roc_auc_score(y_test, test_pred))
print("Matrix: ", confusion_matrix(y_test, test_pred))
print("report: ", classification_report(y_test, test_pred))

ROC-AUC:  0.5210335921225691
Matrix:  [[27930    65]
 [ 1916    89]]
report:                precision    recall  f1-score   support

           0       0.94      1.00      0.97     27995
           1       0.58      0.04      0.08      2005

    accuracy                           0.93     30000
   macro avg       0.76      0.52      0.52     30000
weighted avg       0.91      0.93      0.91     30000

